# Gelişmiş RAG

Temel RAG'ın (parçala → embed → en benzerini getir → cevapla) üzerine üretimde işe yarayan teknikleri ekleyen, uçtan uca bir hat:

| # | Teknik | Ne kazandırır? |
|---|---|---|
| 1 | **Çoklu format yükleme** (TXT, MD, PDF, DOCX) + metadata | Gerçek belgelerle çalışma, filtreleme |
| 2 | **Cümle farkındalıklı chunking** + başlık ekleme (contextual header) | Parçalar anlamlı yerlerden bölünür, bağlamı kaybolmaz |
| 3 | **Hibrit arama**: dense (embedding) + sparse (BM25, Türkçe kök kırpma) | Hem anlamsal hem anahtar kelime eşleşmesi |
| 4 | **Reciprocal Rank Fusion (RRF)** | Farklı arama sonuçlarını güvenle birleştirir |
| 5 | **Sorgu dönüşümleri**: sohbet geçmişinden bağımsız soru, multi-query, HyDE | Kısa / belirsiz sorularda isabet artar |
| 6 | **Cross-encoder reranking** | En alakalı parçalar en üste çıkar |
| 7 | **MMR** (çeşitlilik) | Birbirinin tekrarı olan parçalar elenir |
| 8 | **Komşu parça genişletme** | Cevabın devamı yandaki parçadaysa o da gelir |
| 9 | **Metadata filtreleme** | "Sadece 2025 ve sonrası", "sadece İK" gibi kısıtlar |
| 10 | **Kaynaklı cevap** + "bilgi yok" davranışı | Halüsinasyon azalır, cevap doğrulanabilir |
| 11 | **Sohbet hafızası** | Takip soruları ("peki ya 5 yıldan fazlaysa?") anlaşılır |
| 12 | **Kaydet / yükle** | İndeks bir kez kurulur, tekrar kullanılır |
| 13 | **Değerlendirme**: Hit@k, MRR, sadakat (LLM-as-judge) | Hangi tekniğin ne kadar işe yaradığı ölçülür |

**API anahtarı gerekmez:** Cevapları bilgisayarında çalışan **yerel bir LLM** üretir: [Ollama](https://ollama.com) (önerilen) ya da Hugging Face `transformers`. Hiçbir veri dışarı gönderilmez.

**Dayanıklılık:** Embedding veya reranker modeli indirilemezse kod çökmez, yedek yöntemlere geçer. Yerel LLM bulunamazsa LLM adımları atlanır ve cevaplar belgelerden çıkarılarak (extractive) verilir. Böylece notebook her ortamda baştan sona çalışır.

### Yerel LLM kurulumu (birini seç)
**A) Ollama (önerilen, kendi bilgisayarında)**
1. https://ollama.com adresinden Ollama'yı kur ve aç.
2. Terminalde modeli indir: `ollama pull qwen2.5:7b` (≈4,7 GB, Türkçesi iyi). Daha zayıf bilgisayar için `qwen2.5:3b`.
3. Notebook'ta `llm_backend="ollama"` (veya `"auto"`) bırak.

**B) Hugging Face transformers (Colab / GPU'lu ortam)**
`llm_backend="hf"` yap. Model ilk çalıştırmada otomatik indirilir. Varsayılan `Qwen/Qwen2.5-1.5B-Instruct` CPU'da da çalışır ama yavaştır. GPU varsa `Qwen/Qwen2.5-7B-Instruct` çok daha iyi sonuç verir.

In [ ]:
# Kurulum (bir kez çalıştırman yeterli)
!pip install -q sentence-transformers faiss-cpu scikit-learn pypdf python-docx pandas requests transformers accelerate

## 0. Ayarlar ve içe aktarmalar

In [ ]:
import os, re, math, json, glob, pickle, logging, warnings
from dataclasses import dataclass, field, asdict
from typing import Any, Callable, Dict, List, Optional
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s", force=True)
for noisy in ("httpx", "sentence_transformers", "urllib3", "huggingface_hub", "faiss", "faiss.loader", "transformers"):
    logging.getLogger(noisy).setLevel(logging.ERROR)
log = logging.getLogger("rag")


@dataclass
class RAGConfig:
    # Parçalama
    chunk_size: int = 120            # kelime cinsinden hedef parça boyu
    chunk_overlap: int = 25          # parçalar arası örtüşme (kelime)
    add_title_to_chunk: bool = True  # embedding'e belge başlığını da ekle

    # Modeller (Yükseltilmiş sürüm: BAAI bge modelleri)
    embed_model: str = "BAAI/bge-m3"                                    # Çok gelişmiş, Türkçe performansı yüksek
    rerank_model: str = "BAAI/bge-reranker-v2-m3"                       # Son teknoloji cross-encoder reranker

    # Yerel LLM (API anahtarı gerekmez)
    llm_backend: str = "auto"                     # "ollama" | "hf" | "auto"
    ollama_model: str = "qwen2.5:7b"              # alternatifler: "qwen2.5:3b", "llama3.1:8b", "gemma2:9b"
    ollama_url: str = "http://localhost:11434"
    hf_model: str = "Qwen/Qwen2.5-7B-Instruct"    # Daha yüksek kalite için 7B modeline yükseltildi
    llm_temperature: float = 0.1

    # Arama
    k_dense: int = 20        # dense aramadan alınacak aday sayısı
    k_sparse: int = 20       # BM25'ten alınacak aday sayısı
    rrf_k: int = 60          # RRF sabiti
    dense_weight: float = 1.0
    sparse_weight: float = 1.0
    k_candidates: int = 12   # rerank'e girecek aday sayısı
    k_final: int = 4         # LLM'e verilecek parça sayısı
    mmr_lambda: float = 0.75 # 1 = sadece alaka, 0 = sadece çeşitlilik
    neighbor_window: int = 1 # seçilen parçanın kaç komşusu eklenecek

    # Sorgu dönüşümleri
    use_condense: bool = True     # sohbet geçmişinden bağımsız soru üret
    use_multi_query: bool = True  # sorunun farklı ifadelerini üret
    n_multi_query: int = 3
    use_hyde: bool = False        # varsayımsal cevap üretip onunla ara
    use_rerank: bool = True

cfg = RAGConfig()
cfg

RAGConfig(chunk_size=120, chunk_overlap=25, add_title_to_chunk=True, embed_model='BAAI/bge-m3', rerank_model='BAAI/bge-reranker-v2-m3', llm_backend='auto', ollama_model='qwen2.5:7b', ollama_url='http://localhost:11434', hf_model='Qwen/Qwen2.5-7B-Instruct', llm_temperature=0.1, k_dense=20, k_sparse=20, rrf_k=60, dense_weight=1.0, sparse_weight=1.0, k_candidates=12, k_final=4, mmr_lambda=0.75, neighbor_window=1, use_condense=True, use_multi_query=True, n_multi_query=3, use_hyde=False, use_rerank=True)

## 1. Belgeler ve yükleyiciler
Her belge `text` + `metadata` taşır. Metadata (kaynak, kategori, yıl, sayfa…) hem kaynak göstermede hem filtrelemede kullanılır.

In [ ]:
@dataclass
class Document:
    text: str
    metadata: Dict[str, Any] = field(default_factory=dict)


def clean_text(text: str) -> str:
    text = text.replace("\r", "")
    text = re.sub(r"(\w)-\n(\w)", r"\1\2", text)    # satır sonu tirelemesi: "bil-\ngi" → "bilgi"
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def load_file(path: str) -> List[Document]:
    """TXT, MD, PDF ve DOCX dosyalarını okur. PDF'te her sayfa ayrı belge olur (sayfa numarası metadata'da)."""
    ext = os.path.splitext(path)[1].lower()
    base = {"source": os.path.basename(path), "title": os.path.splitext(os.path.basename(path))[0]}
    if ext in (".txt", ".md"):
        with open(path, encoding="utf-8") as f:
            return [Document(clean_text(f.read()), base)]
    if ext == ".pdf":
        from pypdf import PdfReader
        reader = PdfReader(path)
        return [Document(clean_text(p.extract_text() or ""), {**base, "page": i + 1})
                for i, p in enumerate(reader.pages) if (p.extract_text() or "").strip()]
    if ext == ".docx":
        import docx
        d = docx.Document(path)
        return [Document(clean_text("\n\n".join(p.text for p in d.paragraphs if p.text.strip())), base)]
    raise ValueError(f"Desteklenmeyen dosya türü: {ext}")


def load_directory(path: str, exts=(".txt", ".md", ".pdf", ".docx"), extra_metadata: Optional[dict] = None) -> List[Document]:
    docs = []
    for f in sorted(glob.glob(os.path.join(path, "**", "*"), recursive=True)):
        if f.lower().endswith(exts):
            try:
                for d in load_file(f):
                    d.metadata.update(extra_metadata or {})
                    docs.append(d)
            except Exception as e:
                log.warning(f"{f} okunamadı: {e}")
    log.info(f"{len(docs)} belge yüklendi ({path})")
    return docs

### Örnek bilgi tabanı
Kurgusal bir şirketin (*Örnek Teknoloji A.Ş.*) çalışan el kitabı. Bilgi güvenliği politikasının **eski (2022)** ve **yeni (2025)** sürümü bilerek birlikte konuldu: metadata filtresinin neden önemli olduğunu gösterecek.

Kendi verin için: `documents = load_directory("belgelerim/")`

In [ ]:
documents = [
    Document("""Yıllık İzin Politikası

Çalışanlar kıdemlerine göre yıllık ücretli izin hakkı kazanır. Bir yıldan beş yıla kadar (beş yıl dahil) kıdemi olan çalışanlara yılda 14 iş günü izin verilir. Beş yıldan fazla, on beş yıldan az kıdemi olanlara 20 iş günü, on beş yıl ve üzeri kıdemi olanlara 26 iş günü izin verilir. Elli yaş ve üzeri çalışanlar kıdemlerinden bağımsız olarak en az 20 iş günü izin kullanır.

İzin talepleri en az iki hafta önceden İKSİS portalı üzerinden yapılır ve doğrudan yönetici tarafından onaylanır. Beş iş gününden uzun izinler için ayrıca departman direktörünün onayı gerekir.

Kullanılmayan izinlerin en fazla 5 günü bir sonraki yıla devredilebilir. Devreden izinler 31 Mart tarihine kadar kullanılmazsa yanar.

Babalık izni 10 iş günüdür. Hastalık raporları, rapor başlangıcından itibaren 3 iş günü içinde İnsan Kaynakları birimine iletilmelidir.""",
             {"source": "izin_politikasi.md", "title": "Yıllık İzin Politikası", "category": "ik", "year": 2025}),

    Document("""Uzaktan ve Hibrit Çalışma Politikası

Ofis dışı çalışmaya uygun pozisyonlarda çalışanlar haftada en fazla 3 gün uzaktan çalışabilir. Salı ve perşembe günleri tüm ekipler için ortak ofis günüdür.

Uzaktan çalışma günlerinde 10:00 ile 16:00 arası çekirdek çalışma saatidir; bu saatlerde çalışanın ulaşılabilir olması beklenir. Şirket sistemlerine yalnızca kurumsal VPN üzerinden bağlanılabilir.

Uzaktan çalışan personele aylık 750 TL internet ve ev ofis desteği ödenir. Ergonomik sandalye ve monitör talepleri İK portalından yapılır.

Yurt dışından uzaktan çalışma yılda en fazla 30 takvim günü ile sınırlıdır ve yönetici ile İK onayı gerektirir.""",
             {"source": "uzaktan_calisma.md", "title": "Uzaktan Çalışma Politikası", "category": "ik", "year": 2024}),

    Document("""Seyahat ve Masraf Politikası

İş amaçlı tüm masraflar, harcama tarihinden itibaren en geç 30 gün içinde masraf sistemine girilmelidir. Her masraf kalemi için fatura veya fiş yüklenmesi zorunludur; belgesiz masraflar ödenmez.

Şehir içi yemek masraflarında kişi başı günlük limit 600 TL, şehir dışı seyahatlerde 900 TL'dir. Konaklamada dört yıldız üzeri oteller için önceden finans onayı alınmalıdır.

Taksi masrafları yalnızca 22:00 sonrası veya toplu taşımanın mümkün olmadığı durumlarda karşılanır.

Onay akışı: 5.000 TL'ye kadar olan masrafları doğrudan yönetici onaylar. 5.000 TL üzerindeki masraflar için ek olarak finans direktörünün onayı gerekir. Onaylanan masraflar bir sonraki maaş ödemesiyle birlikte hesaba yatırılır.""",
             {"source": "masraf_politikasi.md", "title": "Seyahat ve Masraf Politikası", "category": "finans", "year": 2025}),

    Document("""Bilgi Güvenliği Politikası (2025)

Tüm kullanıcı parolaları en az 12 karakter uzunluğunda olmalı; büyük harf, küçük harf, rakam ve özel karakter içermelidir. Parolalar 90 günde bir değiştirilir. Tüm kurumsal hesaplarda çok faktörlü kimlik doğrulama (MFA) zorunludur.

Kişisel USB bellek ve harici disklerin şirket bilgisayarlarına takılması yasaktır. Dosya paylaşımı yalnızca kurumsal bulut depolama üzerinden yapılır.

Şüpheli e-postalar (oltalama / phishing) açılmadan guvenlik@ornekteknoloji.com adresine iletilmelidir. Şirket dizüstü bilgisayarının veya telefonunun kaybolması ya da çalınması durumunda en geç 1 saat içinde BT Yardım Masasına bildirim yapılmalıdır.

Veriler dört sınıfa ayrılır: Genel, Kurum İçi, Gizli ve Çok Gizli. Gizli ve Çok Gizli veriler şifrelenmeden e-posta ile gönderilemez.""",
             {"source": "bilgi_guvenligi_2025.md", "title": "Bilgi Güvenliği Politikası", "category": "it", "year": 2025}),

    Document("""Bilgi Güvenliği Politikası (2022 — yürürlükten kaldırıldı)

Kullanıcı parolaları en az 8 karakter olmalıdır. Parolalar 60 günde bir değiştirilir. Çok faktörlü kimlik doğrulama yalnızca yönetici hesapları için zorunludur.

Şüpheli e-postalar BT Yardım Masasına telefonla bildirilir. Cihaz kayıpları 24 saat içinde bildirilmelidir.""",
             {"source": "bilgi_guvenligi_2022.md", "title": "Bilgi Güvenliği Politikası (eski)", "category": "it", "year": 2022}),

    Document("""Eğitim ve Gelişim Politikası

Her çalışanın yıllık kişisel eğitim bütçesi 15.000 TL'dir. Bütçe; kurslar, konferanslar, kitaplar ve çevrim içi eğitim platformları için kullanılabilir. Kullanılmayan bütçe bir sonraki yıla devretmez.

Pozisyonla ilgili profesyonel sertifika sınav ücretleri, ilk denemede başarısız olunsa bile bir kez daha şirket tarafından karşılanır.

Tutarı 30.000 TL'yi aşan eğitimler için eğitim taahhütnamesi imzalanır. Çalışan eğitim bitiminden sonraki 12 ay içinde kendi isteğiyle ayrılırsa eğitim bedelinin kalan süreye düşen kısmını iade eder.

Yeni çalışanlar ilk 30 gün içinde zorunlu oryantasyon ve bilgi güvenliği eğitimlerini tamamlamalıdır.""",
             {"source": "egitim_gelisim.md", "title": "Eğitim ve Gelişim Politikası", "category": "ik", "year": 2025}),
]
print(len(documents), "belge")
pd.DataFrame([d.metadata for d in documents])

6 belge


,source,title,category,year
0,izin_politikasi.md,Yıllık İzin Politikası,ik,2025
1,uzaktan_calisma.md,Uzaktan Çalışma Politikası,ik,2024
2,masraf_politikasi.md,Seyahat ve Masraf Politikası,finans,2025
3,bilgi_guvenligi_2025.md,Bilgi Güvenliği Politikası,it,2025
4,bilgi_guvenligi_2022.md,Bilgi Güvenliği Politikası (eski),it,2022
5,egitim_gelisim.md,Eğitim ve Gelişim Politikası,ik,2025


## 2. Cümle farkındalıklı parçalama (chunking)
Metni rastgele kelime sayısında kesmek yerine **cümleleri** paketler, hedef boyu aşınca yeni parçaya geçer ve son birkaç cümleyi **örtüşme** olarak taşır. Her parçaya belge başlığı eklenir (*contextual chunk header*): "Parolalar 90 günde bir değiştirilir" cümlesi, başlığıyla birlikte embed edilince hangi politikaya ait olduğu da vektöre girer.

In [ ]:
@dataclass
class Chunk:
    id: int
    text: str              # LLM'e gösterilecek metin
    embed_text: str        # vektörleştirilecek metin (başlık eklenmiş)
    doc_id: str
    position: int          # belge içindeki sıra (komşu genişletme için)
    metadata: Dict[str, Any]


_SENT_SPLIT = re.compile(r"(?<=[.!?…])\s+(?=[A-ZÇĞİÖŞÜ0-9\"“(])")

def split_sentences(text: str) -> List[str]:
    sents = []
    for para in re.split(r"\n\s*\n", text):
        para = re.sub(r"\s+", " ", para).strip()
        if para:
            sents.extend(s.strip() for s in _SENT_SPLIT.split(para) if s.strip())
    return sents


def chunk_text(text: str, size: int, overlap: int) -> List[str]:
    # Hedef boydan uzun cümleleri kelime pencerelerine böl
    units = []
    for s in split_sentences(text):
        w = s.split()
        if len(w) > size:
            step = max(1, size - overlap)
            units.extend(" ".join(w[i:i + size]) for i in range(0, len(w), step))
        else:
            units.append(s)

    chunks, cur, cur_len = [], [], 0
    for u in units:
        n = len(u.split())
        if cur and cur_len + n > size:
            chunks.append("\n".join(cur))
            # Örtüşme: son cümleleri bir sonraki parçaya taşı
            keep, kl = [], 0
            for t in reversed(cur):
                tl = len(t.split())
                if kl + tl > overlap:
                    break
                keep.insert(0, t); kl += tl
            cur, cur_len = keep, kl
        cur.append(u); cur_len += n
    if cur:
        chunks.append("\n".join(cur))
    return chunks          # parça içinde her satır bir cümle


def make_chunks(docs: List[Document], cfg: RAGConfig, start_id: int = 0) -> List[Chunk]:
    out, cid = [], start_id
    for d in docs:
        doc_id = d.metadata.get("source", "doc") + (f"#s{d.metadata['page']}" if "page" in d.metadata else "")
        title = d.metadata.get("title", "")
        for pos, t in enumerate(chunk_text(d.text, cfg.chunk_size, cfg.chunk_overlap)):
            et = f"{title}: {t}" if (cfg.add_title_to_chunk and title) else t
            out.append(Chunk(cid, t, et, doc_id, pos, dict(d.metadata)))
            cid += 1
    return out


# Demo için küçük parçalar: örnek belgeler kısa olduğu için
cfg.chunk_size, cfg.chunk_overlap = 45, 12
_demo = make_chunks(documents, cfg)
print(f"{len(_demo)} parça | ortalama {np.mean([len(c.text.split()) for c in _demo]):.0f} kelime\n")
for c in _demo[:3]:
    print(f"[{c.id}] {c.doc_id} / {c.position}: {c.text[:110].replace(chr(10), ' ')}...")

18 parça | ortalama 35 kelime

[0] izin_politikasi.md / 0: Yıllık İzin Politikası Çalışanlar kıdemlerine göre yıllık ücretli izin hakkı kazanır. Bir yıldan beş yıla kada...
[1] izin_politikasi.md / 1: Beş yıldan fazla, on beş yıldan az kıdemi olanlara 20 iş günü, on beş yıl ve üzeri kıdemi olanlara 26 iş günü ...
[2] izin_politikasi.md / 2: İzin talepleri en az iki hafta önceden İKSİS portalı üzerinden yapılır ve doğrudan yönetici tarafından onaylan...


## 3. Sparse arama: Türkçe için BM25
BM25, anahtar kelime eşleşmesini ölçer; ürün kodu, rakam, özel isim gibi embedding'in kaçırabileceği şeyleri yakalar. Türkçe sondan eklemeli olduğu için kelimeleri ilk 5 harfe kırpıyoruz (**F5 stemming**): *izinler, izinlerin, izni* → `izinl / izni` gibi yakın köklerde buluşur. Türkçe büyük/küçük harf dönüşümü (I→ı, İ→i) ayrıca ele alınır.

In [ ]:
TR_STOP = set("""acaba ama ancak artık aslında az bana bazen bazı belki ben beni benim bir biri birkaç birçok biz bu buna
bunda bundan bunu bunun burada çok çünkü da daha de defa diye eğer en gibi hem hep hepsi her hiç için ile ise kaç kez ki kim
mi mu mü mı nasıl ne neden nerede niçin o olan olarak oldu olduğu olmak olur ona ondan onlar onu onun öyle sanki şey siz şu
tüm ve veya ya yani hangi peki var yok""".split())

def tr_lower(s: str) -> str:
    return s.replace("I", "ı").replace("İ", "i").lower()

def tokenize(text: str, stem: int = 5) -> List[str]:
    toks = re.findall(r"[0-9a-zçğıöşüâîû]+", tr_lower(text))
    return [t[:stem] if (stem and not t.isdigit()) else t for t in toks if t not in TR_STOP and len(t) > 1]


class BM25:
    def __init__(self, k1: float = 1.5, b: float = 0.75):
        self.k1, self.b = k1, b

    def fit(self, corpus_tokens: List[List[str]]):
        self.N = len(corpus_tokens)
        self.doc_len = np.array([len(d) for d in corpus_tokens], dtype=float)
        self.avgdl = self.doc_len.mean() if self.N else 1.0
        df = Counter(t for d in corpus_tokens for t in set(d))
        self.idf = {t: math.log(1 + (self.N - n + 0.5) / (n + 0.5)) for t, n in df.items()}
        self.inverted = defaultdict(list)                      # terim → [(parça, frekans)]
        for i, d in enumerate(corpus_tokens):
            for t, f in Counter(d).items():
                self.inverted[t].append((i, f))
        return self

    def scores(self, query_tokens: List[str]) -> np.ndarray:
        s = np.zeros(self.N)
        for t in set(query_tokens):
            idf = self.idf.get(t)
            if idf is None:
                continue
            for i, f in self.inverted[t]:
                s[i] += idf * f * (self.k1 + 1) / (f + self.k1 * (1 - self.b + self.b * self.doc_len[i] / self.avgdl))
        return s

print(tokenize("İzin taleplerinin İKSİS portalından yapılması zorunludur."))

['izin', 'talep', 'iksis', 'porta', 'yapıl', 'zorun']


## 4. Dense arama: embedding + vektör indeksi
`multilingual-e5` modelleri sorgu ve belge için farklı önek ister (`query:` / `passage:`). Model indirilemezse karakter n-gram tabanlı bir yedek vektörleştiriciye geçilir. İndeks olarak FAISS kullanılır; kurulu değilse NumPy ile aynı hesap yapılır.

In [ ]:
class Embedder:
    def __init__(self, model_name: str):
        self.model_name = model_name
        try:
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer(model_name)
            self.backend = "sentence-transformers"
            self.is_e5 = "e5" in model_name.lower()
        except Exception as e:
            log.warning(f"Embedding modeli yüklenemedi ({type(e).__name__}). Yedek: karakter n-gram vektörleri.")
            from sklearn.feature_extraction.text import HashingVectorizer
            self.model = HashingVectorizer(analyzer="char_wb", ngram_range=(3, 5), n_features=2**14,
                                           alternate_sign=False, norm="l2")
            self.backend = "char-ngram (yedek)"
            self.is_e5 = False

    def _encode(self, texts: List[str], prefix: str) -> np.ndarray:
        if self.backend == "sentence-transformers":
            if self.is_e5:
                texts = [prefix + t for t in texts]
            v = self.model.encode(texts, normalize_embeddings=True, batch_size=32, show_progress_bar=False)
            return np.asarray(v, dtype="float32")
        return self.model.transform([tr_lower(t) for t in texts]).toarray().astype("float32")

    def embed_documents(self, texts: List[str]) -> np.ndarray:
        return self._encode(texts, "passage: ")

    def embed_query(self, text: str) -> np.ndarray:
        return self._encode([text], "query: ")[0]


class VectorIndex:
    """Normalize vektörlerde iç çarpım = kosinüs benzerliği."""
    def __init__(self):
        self.vectors = None
        self.faiss_index = None

    def add(self, vecs: np.ndarray):
        self.vectors = vecs if self.vectors is None else np.vstack([self.vectors, vecs])
        try:
            import faiss
            self.faiss_index = faiss.IndexFlatIP(self.vectors.shape[1])
            self.faiss_index.add(self.vectors)
        except ImportError:
            self.faiss_index = None

    def search(self, q: np.ndarray, k: int, allowed: Optional[np.ndarray] = None):
        """En benzer k parçanın (id, skor) listesi. `allowed`: filtreden geçen parça maskesi."""
        if allowed is None and self.faiss_index is not None:
            D, I = self.faiss_index.search(q[None, :].astype("float32"), min(k, len(self.vectors)))
            return [(int(i), float(d)) for i, d in zip(I[0], D[0]) if i >= 0]
        s = self.vectors @ q
        if allowed is not None:
            s = np.where(allowed, s, -np.inf)
        top = np.argsort(-s)[:k]
        return [(int(i), float(s[i])) for i in top if np.isfinite(s[i])]

## 5. Birleştirme, reranking ve çeşitlilik

- **Cross-encoder reranker:** Soru ve parçayı *birlikte* okuyup alaka puanı verir. Embedding'den çok daha isabetlidir ama yavaştır, o yüzden sadece ilk adaylara uygulanır.
- **MMR:** Alakalı ama birbirine benzemeyen parçaları seçer.

In [ ]:
def reciprocal_rank_fusion(ranked_lists: List[List[int]], k: int = 60, weights: Optional[List[float]] = None) -> List[tuple]:
    scores = defaultdict(float)
    weights = weights or [1.0] * len(ranked_lists)
    for w, lst in zip(weights, ranked_lists):
        for rank, idx in enumerate(lst):
            scores[idx] += w / (k + rank + 1)
    return sorted(scores.items(), key=lambda x: -x[1])


class Reranker:
    def __init__(self, model_name: str):
        try:
            from sentence_transformers import CrossEncoder
            self.model = CrossEncoder(model_name, max_length=512)
            self.available = True
        except Exception as e:
            log.warning(f"Reranker yüklenemedi ({type(e).__name__}). Reranking atlanacak, RRF sırası kullanılacak.")
            self.model, self.available = None, False

    def score(self, query: str, texts: List[str]) -> np.ndarray:
        return np.asarray(self.model.predict([(query, t) for t in texts], show_progress_bar=False), dtype=float)


def mmr_select(candidates: List[int], relevance: Dict[int, float], vectors: np.ndarray, k: int, lam: float) -> List[int]:
    if not candidates:
        return []
    r = np.array([relevance[i] for i in candidates])
    r = (r - r.min()) / (r.max() - r.min() + 1e-9)          # alaka puanlarını 0–1'e ölçekle
    rel = dict(zip(candidates, r))
    selected, pool = [], list(candidates)
    while pool and len(selected) < k:
        if not selected:
            best = max(pool, key=lambda i: rel[i])
        else:
            S = vectors[selected]
            best = max(pool, key=lambda i: lam * rel[i] - (1 - lam) * float(np.max(S @ vectors[i])))
        selected.append(best)
        pool.remove(best)
    return selected

## 6. LLM ve sorgu dönüşümleri
- **Condense:** Sohbet geçmişini kullanarak takip sorusunu tek başına anlaşılır hale getirir (*"peki 5 yıldan fazlaysa?"* → *"5 yıldan fazla kıdemi olanın yıllık izni kaç gün?"*).
- **Multi-query:** Aynı soruyu farklı kelimelerle 3 kez yazar; her biriyle ayrı arama yapılır, sonuçlar RRF ile birleşir.
- **HyDE:** Önce varsayımsal bir cevap paragrafı yazdırır, onunla arar. Soru ile belgenin dili çok farklıysa işe yarar.

Yerel LLM yoksa bu adımlar sessizce atlanır. Küçük modellerde (1–3B) multi-query bazen zayıf varyantlar üretebilir; o durumda `use_multi_query=False` yap.

In [ ]:
class LLM:
    """API anahtarı gerektirmeyen, bilgisayarında çalışan yerel LLM.

    backend:
      "ollama" → Ollama sunucusu (önerilen: kurulumu kolay, hızlı, GPU/CPU otomatik)
      "hf"     → Hugging Face transformers; model notebook'un içinde çalışır (Colab için uygun)
      "auto"   → önce Ollama'yı dener, bulamazsa transformers'a geçer
    Hiçbiri çalışmazsa LLM adımları atlanır, cevaplar belgelerden alıntılanır.
    """
    def __init__(self, cfg: "RAGConfig"):
        self.cfg, self.backend, self.name = cfg, None, None
        order = ["ollama", "hf"] if cfg.llm_backend == "auto" else [cfg.llm_backend]
        for b in order:
            try:
                getattr(self, f"_init_{b}")()
                self.backend = b
                break
            except Exception as e:
                log.warning(f"LLM ({b}) kullanılamıyor: {type(e).__name__}: {str(e)[:150]}")
        if self.backend:
            log.info(f"LLM hazır: {self.backend} / {self.name}")
        else:
            log.warning("Yerel LLM bulunamadı: sorgu dönüşümleri atlanacak, cevaplar belgelerden alıntılanacak.")

    @property
    def available(self) -> bool:
        return self.backend is not None

    # ---------- Ollama ----------
    def _init_ollama(self):
        import requests
        r = requests.get(f"{self.cfg.ollama_url}/api/tags", timeout=3)
        r.raise_for_status()
        installed = {m["name"] for m in r.json().get("models", [])}
        want = self.cfg.ollama_model
        if want not in installed and f"{want}:latest" not in installed:
            raise RuntimeError(f"'{want}' modeli indirilmemiş. Terminalde çalıştır: ollama pull {want}  "
                               f"(yüklü modeller: {sorted(installed) or 'yok'})")
        self.name = want

    def _complete_ollama(self, messages: List[dict], max_tokens: int) -> str:
        import requests
        r = requests.post(f"{self.cfg.ollama_url}/api/chat", timeout=600, json={
            "model": self.name, "messages": messages, "stream": False,
            "options": {"temperature": self.cfg.llm_temperature, "num_predict": max_tokens, "num_ctx": 8192},
        })
        r.raise_for_status()
        return r.json()["message"]["content"]

    # ---------- Hugging Face transformers ----------
    def _init_hf(self):
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer
        name = self.cfg.hf_model
        self.tokenizer = AutoTokenizer.from_pretrained(name)
        self.model = AutoModelForCausalLM.from_pretrained(
            name, torch_dtype="auto", device_map="auto" if torch.cuda.is_available() else None)
        self.model.eval()
        self.name = name

    def _complete_hf(self, messages: List[dict], max_tokens: int) -> str:
        import torch
        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(text, return_tensors="pt").to(self.model.device)
        with torch.no_grad():
            out = self.model.generate(**inputs, max_new_tokens=max_tokens, do_sample=False,
                                      repetition_penalty=1.05, pad_token_id=self.tokenizer.eos_token_id)
        return self.tokenizer.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    # ---------- Ortak arayüz ----------
    def complete(self, prompt: str, system: Optional[str] = None, max_tokens: int = 800) -> str:
        messages = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": prompt}]
        out = getattr(self, f"_complete_{self.backend}")(messages, max_tokens)
        return re.sub(r"<think>.*?</think>", "", out, flags=re.S).strip()   # "düşünen" modellerin iç notlarını at


def condense_question(llm: LLM, history: List[dict], question: str) -> str:
    if not history or not llm.available:
        return question
    h = "\n".join(f"{m['role']}: {m['content']}" for m in history[-6:])
    prompt = f"""Aşağıdaki sohbet geçmişini ve takip sorusunu kullanarak, geçmişe bakmadan tek başına anlaşılabilecek bir arama sorusu yaz.
Sadece soruyu yaz, açıklama ekleme.

Sohbet geçmişi:
{h}

Takip sorusu: {question}
Bağımsız soru:"""
    return llm.complete(prompt, max_tokens=150).strip().strip('"')


def generate_multi_queries(llm: LLM, question: str, n: int = 3) -> List[str]:
    if not llm.available:
        return []
    prompt = f"""Bir belge arama sistemi için aşağıdaki soruyu {n} farklı şekilde yeniden yaz.
Farklı eş anlamlı kelimeler ve ifadeler kullan. Her satıra bir soru yaz; numara veya açıklama ekleme.

Soru: {question}"""
    lines = [re.sub(r"^[\-\d.)\s]+", "", l).strip() for l in llm.complete(prompt, max_tokens=300).splitlines()]
    return [l for l in lines if l][:n]


def generate_hyde(llm: LLM, question: str) -> Optional[str]:
    if not llm.available:
        return None
    return llm.complete(f"Aşağıdaki soruya şirket politikası belgesinde geçebilecek kısa bir paragrafla cevap yaz:\n\n{question}",
                        max_tokens=200)

## 7. Metadata filtreleme
Filtre bir sözlüktür. Değer düz bir değer (eşitlik), bir liste (içinde mi) ya da bir fonksiyon olabilir:
```python
{"category": "ik"}                          # eşitlik
{"category": ["ik", "finans"]}              # listeden biri
{"year": lambda y: y is not None and y >= 2025}   # özel koşul
```

In [ ]:
def matches(meta: Dict[str, Any], filters: Optional[Dict[str, Any]]) -> bool:
    if not filters:
        return True
    for key, cond in filters.items():
        val = meta.get(key)
        if callable(cond):
            if not cond(val):
                return False
        elif isinstance(cond, (list, tuple, set)):
            if val not in cond:
                return False
        elif val != cond:
            return False
    return True

## 8. Hepsi bir arada: `AdvancedRAG`
`retrieve()` adımları:
1. Soruyu sohbet geçmişine göre bağımsız hale getir (condense)
2. Multi-query / HyDE ile sorgu varyantları üret
3. Her varyant için **dense** + **BM25** arama (filtreli)
4. Tüm listeleri **RRF** ile birleştir
5. İlk adayları **cross-encoder** ile yeniden sırala
6. **MMR** ile çeşitli `k_final` parça seç
7. Seçilen parçaları **komşularıyla genişlet**

`mode` parametresiyle (`"bm25"`, `"dense"`, `"hybrid"`, `"full"`) her tekniğin katkısını ayrı ayrı ölçebiliriz.

In [ ]:
SYSTEM_PROMPT = """Sen Örnek Teknoloji A.Ş. çalışanlarına şirket politikaları hakkında yardım eden bir asistansın.
Kurallar:
- YALNIZCA verilen bağlamdaki bilgileri kullan; kendi genel bilginle boşluk doldurma.
- Her bilginin sonunda kaynağını [1], [2] biçiminde belirt.
- Bağlamda cevap yoksa tam olarak şunu yaz: "Bu soruya belgelerde yeterli bilgi bulunamadı."
- Kaynaklar çelişiyorsa en güncel yıla ait olanı esas al ve çelişkiyi belirt.
- Kısa, net ve Türkçe cevap ver."""


class AdvancedRAG:
    def __init__(self, cfg: RAGConfig):
        self.cfg = cfg
        self.chunks: List[Chunk] = []
        self.embedder = Embedder(cfg.embed_model)
        self.reranker = Reranker(cfg.rerank_model) if cfg.use_rerank else None
        self.llm = LLM(cfg)
        self.index = VectorIndex()
        self.bm25 = BM25()
        self._by_doc: Dict[str, Dict[int, int]] = {}

    # ---------- İndeksleme ----------
    def add_documents(self, docs: List[Document]):
        new = make_chunks(docs, self.cfg, start_id=len(self.chunks))
        if not new:
            return
        self.index.add(self.embedder.embed_documents([c.embed_text for c in new]))
        self.chunks.extend(new)
        self._rebuild_sparse()
        log.info(f"{len(docs)} belge → {len(new)} parça eklendi (toplam {len(self.chunks)}) | embedding: {self.embedder.backend}")

    def _rebuild_sparse(self):
        self.bm25.fit([tokenize(c.embed_text) for c in self.chunks])
        self._by_doc = defaultdict(dict)
        for c in self.chunks:
            self._by_doc[c.doc_id][c.position] = c.id

    # ---------- Arama ----------
    def _allowed_mask(self, filters):
        if not filters:
            return None
        return np.array([matches(c.metadata, filters) for c in self.chunks])

    def _dense(self, query: str, allowed, k: int, vec: Optional[np.ndarray] = None) -> List[int]:
        q = vec if vec is not None else self.embedder.embed_query(query)
        return [i for i, _ in self.index.search(q, k, allowed)]

    def _sparse(self, query: str, allowed, k: int) -> List[int]:
        s = self.bm25.scores(tokenize(query))
        if allowed is not None:
            s = np.where(allowed, s, -np.inf)
        top = np.argsort(-s)[:k]
        return [int(i) for i in top if s[i] > 0]

    def retrieve(self, question: str, filters: Optional[dict] = None, history: Optional[List[dict]] = None,
                 mode: str = "full", k: Optional[int] = None) -> Dict[str, Any]:
        cfg, k = self.cfg, k or self.cfg.k_final
        trace = {"question": question, "mode": mode, "filters": list((filters or {}).keys())}
        allowed = self._allowed_mask(filters)

        # 1) Sorgu dönüşümleri (sadece "full" modda)
        query = question
        variants = [question]
        hyde_vec = None
        if mode == "full":
            if cfg.use_condense and history:
                query = condense_question(self.llm, history, question)
                variants = [query]
            if cfg.use_multi_query:
                variants += generate_multi_queries(self.llm, query, cfg.n_multi_query)
            if cfg.use_hyde:
                h = generate_hyde(self.llm, query)
                if h:
                    hyde_vec = self.embedder.embed_documents([h])[0]
        trace.update(standalone_query=query, query_variants=variants)

        # 2) Arama + 3) RRF
        lists, weights = [], []
        for v in variants:
            if mode in ("dense", "hybrid", "full"):
                lists.append(self._dense(v, allowed, cfg.k_dense)); weights.append(cfg.dense_weight)
            if mode in ("bm25", "hybrid", "full"):
                lists.append(self._sparse(v, allowed, cfg.k_sparse)); weights.append(cfg.sparse_weight)
        if hyde_vec is not None:
            lists.append(self._dense("", allowed, cfg.k_dense, vec=hyde_vec)); weights.append(cfg.dense_weight)

        fused = reciprocal_rank_fusion(lists, cfg.rrf_k, weights)
        if mode in ("bm25", "dense", "hybrid"):          # karşılaştırma modları: ham sıralama
            ids = [i for i, _ in fused[:k]]
            return {"hits": [self._hit(i, s) for i, s in fused[:k]], "ids": ids, "trace": trace}

        candidates = [i for i, _ in fused[:cfg.k_candidates]]
        relevance = dict(fused)

        # 4) Reranking
        if self.reranker is not None and self.reranker.available and candidates:
            rs = self.reranker.score(query, [self.chunks[i].text for i in candidates])
            relevance = dict(zip(candidates, rs.tolist()))
            candidates = sorted(candidates, key=lambda i: -relevance[i])
            trace["reranked"] = True

        # 5) MMR
        selected = mmr_select(candidates, relevance, self.index.vectors, k, cfg.mmr_lambda)
        hits = [self._hit(i, relevance[i]) for i in selected]

        # 6) Komşu genişletme
        for h in hits:
            h["context"] = self._expand(self.chunks[h["id"]], cfg.neighbor_window)
        return {"hits": hits, "ids": selected, "trace": trace}

    def _hit(self, i: int, score: float) -> Dict[str, Any]:
        c = self.chunks[i]
        return {"id": i, "score": float(score), "doc_id": c.doc_id, "text": c.text, "context": c.text, "metadata": c.metadata}

    def _expand(self, c: Chunk, window: int) -> str:
        if window <= 0:
            return c.text
        positions = self._by_doc[c.doc_id]
        seen, sents = set(), []
        for p in range(c.position - window, c.position + window + 1):
            if p in positions:
                for s in self.chunks[positions[p]].text.split("\n"):   # örtüşen cümleleri tekrar etme
                    if s not in seen:
                        seen.add(s); sents.append(s)
        return "\n".join(sents)

    # ---------- Cevap üretimi ----------
    def build_prompt(self, question: str, hits: List[dict]) -> str:
        blocks = []
        for n, h in enumerate(hits, 1):
            m = h["metadata"]
            src = f"{m.get('title', h['doc_id'])} | {m.get('source', '')}" + (f" | yıl {m['year']}" if "year" in m else "") \
                  + (f" | sayfa {m['page']}" if "page" in m else "")
            blocks.append(f"[{n}] ({src})\n{h['context']}")
        return f"### Bağlam\n" + "\n\n".join(blocks) + f"\n\n### Soru\n{question}"

    def _extractive_answer(self, question: str, hits: List[dict], n_sent: int = 3, min_coverage: float = 0.15) -> str:
        """LLM yoksa: sorunun (IDF ağırlıklı) en büyük kısmını kapsayan cümleleri kaynaklarıyla döndür.
        Not: Kelime örtüşmesine dayanır; "bilgi yok" tespiti kabadır. Güvenilir ret için LLM gerekir."""
        q = set(tokenize(question))
        max_idf = max(self.bm25.idf.values(), default=1.0)
        idf = lambda t: self.bm25.idf.get(t, max_idf)        # belgelerde hiç geçmeyen terim = en nadir terim
        total = sum(idf(t) for t in q) or 1.0
        best: Dict[str, tuple] = {}
        for n, h in enumerate(hits, 1):
            for s in h["context"].split("\n"):
                if len(s.split()) < 4 or s in best or not re.search(r"[.!?:;)]$", s):   # başlık ve tekrarları atla
                    continue
                cov = sum(idf(t) for t in q & set(tokenize(s))) / total
                if cov > 0:
                    best[s] = (cov, n)
        ranked = sorted(best.items(), key=lambda x: -x[1][0])
        if not ranked or ranked[0][1][0] < min_coverage:
            return "Bu soruya belgelerde yeterli bilgi bulunamadı."
        top = ranked[0][1][0]
        picked = [(s, n) for s, (c, n) in ranked[:n_sent] if c >= 0.5 * top]
        return "(LLM yok, belgelerden alıntı) " + " ".join(f"{s} [{n}]" for s, n in picked)

    def answer(self, question: str, filters: Optional[dict] = None, history: Optional[List[dict]] = None,
               verbose: bool = True) -> Dict[str, Any]:
        r = self.retrieve(question, filters=filters, history=history)
        hits, trace = r["hits"], r["trace"]
        q = trace.get("standalone_query", question)
        if not hits:
            ans = "Bu soruya belgelerde yeterli bilgi bulunamadı."
        elif self.llm.available:
            ans = self.llm.complete(self.build_prompt(q, hits), system=SYSTEM_PROMPT)
        else:
            ans = self._extractive_answer(q, hits)
        result = {"question": question, "answer": ans, "sources": hits, "trace": trace}
        if verbose:
            show(result)
        return result

    # ---------- Kaydet / yükle ----------
    def save(self, path: str):
        os.makedirs(path, exist_ok=True)
        np.save(os.path.join(path, "vectors.npy"), self.index.vectors)
        with open(os.path.join(path, "chunks.pkl"), "wb") as f:
            pickle.dump({"chunks": [asdict(c) for c in self.chunks], "cfg": asdict(self.cfg),
                         "embed_backend": self.embedder.backend}, f)
        log.info(f"İndeks kaydedildi → {path}/")

    @classmethod
    def load(cls, path: str) -> "AdvancedRAG":
        with open(os.path.join(path, "chunks.pkl"), "rb") as f:
            data = pickle.load(f)
        rag = cls(RAGConfig(**data["cfg"]))
        if rag.embedder.backend != data["embed_backend"]:
            raise RuntimeError("Kayıtlı indeks farklı bir embedding yöntemiyle oluşturulmuş; yeniden indeksleyin.")
        rag.chunks = [Chunk(**c) for c in data["chunks"]]
        rag.index.add(np.load(os.path.join(path, "vectors.npy")))
        rag._rebuild_sparse()
        log.info(f"İndeks yüklendi: {len(rag.chunks)} parça")
        return rag


def show(result: Dict[str, Any]):
    t = result["trace"]
    print(f"❓ {result['question']}")
    if t.get("standalone_query") and t["standalone_query"] != result["question"]:
        print(f"↪ Bağımsız soru: {t['standalone_query']}")
    if len(t.get("query_variants", [])) > 1:
        print("↪ Sorgu varyantları:", " | ".join(t["query_variants"][1:]))
    print(f"\n💬 {result['answer']}\n")
    print("📚 Kaynaklar:")
    for n, h in enumerate(result["sources"], 1):
        print(f"  [{n}] {h['doc_id']:<26} skor={h['score']:+.3f}  {h['text'][:70].replace(chr(10), ' ')}...")
    print("-" * 100)

## 9. İndeksle ve sor

In [ ]:
rag = AdvancedRAG(cfg)
rag.add_documents(documents)
print("Embedding:", rag.embedder.backend, "| Reranker:", "var" if rag.reranker and rag.reranker.available else "yok",
      "| LLM:", f"{rag.llm.backend} / {rag.llm.name}" if rag.llm.available else "yok (extractive mod)")

INFO | TensorFlow version 2.21.0 available.
INFO | JAX version 0.11.1 available.
WARNING | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

... (29 outputs left out)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

INFO | LLM hazır: hf / Qwen/Qwen2.5-7B-Instruct
INFO | 6 belge → 18 parça eklendi (toplam 18) | embedding: sentence-transformers


Embedding: sentence-transformers | Reranker: var | LLM: hf / Qwen/Qwen2.5-7B-Instruct


In [ ]:
_ = rag.answer("Yıllık izin talebini nereden ve ne kadar önce yapmalıyım?")
_ = rag.answer("2000 TL'lik bir masrafı kim onaylar, ne zamana kadar girmem gerekir?")

❓ Yıllık izin talebini nereden ve ne kadar önce yapmalıyım?
↪ Sorgu varyantları: **Açıklama:** Çalışanların yıllık izin taleplerini nasıl ve ne zaman vermeleri gerektiği hakkında bilgi sağlayın. | Yıllık izin taleplerini hangi süreçten ve ne kadar öncesinde sunmalıyım? | **Açıklama:** İşgücü, yıllık izinlerini hangi adımdan ve ne zaman başvurmalı olduğunu öğrenmek istiyor.

💬 Yıllık izin talepleri en az iki hafta öncesi İKSİS portalı üzerinden yapılır ve doğrudan yönetici tarafından onaylanır. [1]

📚 Kaynaklar:
  [1] izin_politikasi.md         skor=+0.867  İzin talepleri en az iki hafta önceden İKSİS portalı üzerinden yapılır...
  [2] izin_politikasi.md         skor=+0.100  Yıllık İzin Politikası Çalışanlar kıdemlerine göre yıllık ücretli izin...
  [3] bilgi_guvenligi_2025.md    skor=+0.000  Bilgi Güvenliği Politikası (2025) Tüm kullanıcı parolaları en az 12 ka...
  [4] masraf_politikasi.md       skor=+0.000  Konaklamada dört yıldız üzeri oteller için önceden finans onayı alınma...
---

### Metadata filtresi: eski ve yeni politika
Filtre olmadan arama, yürürlükten kalkmış 2022 politikasını da getirebilir. Yıl filtresiyle sadece güncel belgeler aranır.

In [ ]:
q = "Parolam en az kaç karakter olmalı ve ne sıklıkla değiştirmeliyim?"
print("=== Filtresiz ===")
_ = rag.answer(q)
print("=== Sadece 2025 ve sonrası ===")
_ = rag.answer(q, filters={"year": lambda y: y is not None and y >= 2025})

=== Filtresiz ===
❓ Parolam en az kaç karakter olmalı ve ne sıklıkla değiştirmeliyim?
↪ Sorgu varyantları: **Açıklama:** Parola uzunluğunu ve düzenli olarak değiştirme frekansını belirtin. | Parolanızın en az kaç karakter olması gerekiyor ve ne sıklıkta değiştirmelisiniz? | **Açıklama:** Parola minimum uzunluğu ve düzenli olarak değiştirme dönemini sorun.

💬 Parolam en az 12 karakter uzunluğunda olmalı ve 90 günde bir değiştirilmelidir. [1]

📚 Kaynaklar:
  [1] bilgi_guvenligi_2025.md    skor=+0.980  Bilgi Güvenliği Politikası (2025) Tüm kullanıcı parolaları en az 12 ka...
  [2] bilgi_guvenligi_2022.md    skor=+0.959  Bilgi Güvenliği Politikası (2022 — yürürlükten kaldırıldı) Kullanıcı p...
  [3] masraf_politikasi.md       skor=+0.000  Konaklamada dört yıldız üzeri oteller için önceden finans onayı alınma...
  [4] uzaktan_calisma.md         skor=+0.000  Uzaktan ve Hibrit Çalışma Politikası Ofis dışı çalışmaya uygun pozisyo...
-------------------------------------------------------------

In [ ]:
# Kategori filtresi: sadece İK belgelerinde ara
_ = rag.answer("Eğitim bütçem ne kadar, kullanmazsam devreder mi?", filters={"category": "ik"})

❓ Eğitim bütçem ne kadar, kullanmazsam devreder mi?
↪ Sorgu varyantları: **Açıklama:** Eğitim bütçenizin kullanım süresi hakkında bilgi almak istiyorum. | Soru: Eğitim bütçem sonlanır mı? | **Açıklama:** Eğitim bütçesinin bitiş condan kaçış olanağı var mı?

💬 Eğitim bütçeniz her yıl 15.000 TL'dir. Kullanılmayan bütçe bir sonraki yıla devretmez. Bu bilgi [1] ve [2] kaynaklarından elde edilmiştir.

📚 Kaynaklar:
  [1] egitim_gelisim.md          skor=+0.986  Eğitim ve Gelişim Politikası Her çalışanın yıllık kişisel eğitim bütçe...
  [2] egitim_gelisim.md          skor=+0.963  Kullanılmayan bütçe bir sonraki yıla devretmez. Pozisyonla ilgili prof...
  [3] izin_politikasi.md         skor=+0.011  İzin talepleri en az iki hafta önceden İKSİS portalı üzerinden yapılır...
  [4] uzaktan_calisma.md         skor=+0.000  Şirket sistemlerine yalnızca kurumsal VPN üzerinden bağlanılabilir. Uz...
----------------------------------------------------------------------------------------------------


### Belgelerde olmayan bir soru
İyi bir RAG sistemi bilmediğini söylemeli, uydurmamalı. Bu davranışı asıl sağlayan, sistem prompt'undaki kuraldır; o yüzden **LLM gerekir**. LLM yokken çalışan alıntı modu sadece kelime örtüşmesine bakar: hiç alakasız sorularda "bilgi yok" der, ama kısmen benzeyen sorularda yakın cümleleri döndürebilir.

In [ ]:
_ = rag.answer("Şirketin yemek kartı hangi bankayla anlaşmalı?")
_ = rag.answer("Ofise köpeğimi getirebilir miyim?")

❓ Şirketin yemek kartı hangi bankayla anlaşmalı?
↪ Sorgu varyantları: #1: Şirketin yemek kartı ile anlaşma yaptığı banka hangisidir? | #2: Hangi bankanın şirketin yemek kartı ile anlaşma yaptığı bilgisi bulunmaktadır? | #3: Şirketin yemek kartı için hangi bankaya ödeme hizmetleri sağlayıcısı olarak seçilmiş?

💬 Bu soruya belgelerde yeterli bilgi bulunamadı.

📚 Kaynaklar:
  [1] masraf_politikasi.md       skor=+0.000  Her masraf kalemi için fatura veya fiş yüklenmesi zorunludur; belgesiz...
  [2] uzaktan_calisma.md         skor=+0.000  Şirket sistemlerine yalnızca kurumsal VPN üzerinden bağlanılabilir. Uz...
  [3] masraf_politikasi.md       skor=+0.000  Seyahat ve Masraf Politikası İş amaçlı tüm masraflar, harcama tarihind...
  [4] bilgi_guvenligi_2025.md    skor=+0.000  Şüpheli e-postalar (oltalama / phishing) açılmadan guvenlik@ornektekno...
----------------------------------------------------------------------------------------------------
❓ Ofise köpeğimi getirebilir miyim?
↪ Sorgu v

## 10. Sohbet hafızası
Takip soruları önceki konuşmaya göre bağımsız hale getirilir (bu adım LLM gerektirir).

In [ ]:
class RAGChat:
    def __init__(self, rag: AdvancedRAG, max_turns: int = 6, **default_kwargs):
        self.rag, self.max_turns, self.kw = rag, max_turns, default_kwargs
        self.history: List[dict] = []

    def ask(self, question: str, **kwargs) -> str:
        res = self.rag.answer(question, history=self.history, **{**self.kw, **kwargs})
        self.history += [{"role": "user", "content": question}, {"role": "assistant", "content": res["answer"]}]
        self.history = self.history[-2 * self.max_turns:]
        return res["answer"]

    def reset(self):
        self.history = []


chat = RAGChat(rag)
_ = chat.ask("Uzaktan çalışma kuralları neler?")
_ = chat.ask("Peki yurt dışından da çalışabilir miyim?")

❓ Uzaktan çalışma kuralları neler?
↪ Sorgu varyantları: Numara: 1 | Soru: Uzaktan çalışma koşulları ne içerir? | Numara: 2

💬 Uzaktan çalışma kuralları şunlardır:

- Ofis dışı çalışmaya uygun pozisyonlarda çalışanlar haftada en fazla 3 gün uzaktan çalışabilir.
- Salı ve perşembe günleri tüm ekipler için ortak ofis günüdür.
- Uzaktan çalışma günlerinde 10:00 ile 16:00 arası çekirdek çalışma saatidir; bu saatlerde çalışanın ulaşılabilir olması beklenir.
- Şirket sistemlerine yalnızca kurumsal VPN üzerinden bağlanılabilir.
- Uzaktan çalışan personele aylık 750 TL internet ve ev ofis desteği ödenir.
- Ergonomik sandalye ve monitör talepleri İK portalından yapılır.
- Yurt dışından uzaktan çalışma yılda en fazla 30 takvim günü ile sınırlıdır ve yönetici ile İK onayı gerektirir.

📚 Kaynaklar:
  [1] uzaktan_calisma.md         skor=+0.945  Uzaktan ve Hibrit Çalışma Politikası Ofis dışı çalışmaya uygun pozisyo...
  [2] uzaktan_calisma.md         skor=+0.656  Şirket sistemlerine yalnızca kurumsal

## 11. Kaydet ve yeniden yükle

In [ ]:
rag.save("rag_index")
rag2 = AdvancedRAG.load("rag_index")
_ = rag2.answer("Dizüstü bilgisayarım çalınırsa ne kadar sürede bildirmeliyim?",
                filters={"year": lambda y: y is not None and y >= 2025})

INFO | İndeks kaydedildi → rag_index/


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

WARNING | LLM (ollama) kullanılamıyor: ConnectionError: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/tags (Caused by NewConnectionError('<urllib3.connection.HTTPConn


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

INFO | LLM hazır: hf / Qwen/Qwen2.5-7B-Instruct
INFO | İndeks yüklendi: 18 parça


❓ Dizüstü bilgisayarım çalınırsa ne kadar sürede bildirmeliyim?
↪ Sorgu varyantları: Numara: 1 | Soru: Dizüstü bilgisayarınız kaybolursa, kaç saniye içinde bilgi verebilirsiniz? | Numara: 2

💬 En geç 1 saat içinde BT Yardım Masasına bildirim yapmalısınız. [1]

📚 Kaynaklar:
  [1] bilgi_guvenligi_2025.md    skor=+0.977  Şüpheli e-postalar (oltalama / phishing) açılmadan guvenlik@ornektekno...
  [2] masraf_politikasi.md       skor=+0.000  Her masraf kalemi için fatura veya fiş yüklenmesi zorunludur; belgesiz...
  [3] izin_politikasi.md         skor=+0.000  Yıllık İzin Politikası Çalışanlar kıdemlerine göre yıllık ücretli izin...
  [4] egitim_gelisim.md          skor=+0.000  Tutarı 30.000 TL'yi aşan eğitimler için eğitim taahhütnamesi imzalanır...
----------------------------------------------------------------------------------------------------


## 12. Değerlendirme
Her soru için doğru cevabı içeren belgeyi biliyoruz. Dört arama stratejisini karşılaştırıyoruz:

- **Hit@k:** Doğru belge ilk k sonuç içinde mi?
- **MRR:** Doğru belgenin ilk göründüğü sıranın tersi (1. sıra = 1, 2. sıra = 0,5 …). Ortalaması alınır.

Değerlendirme kendi sorularınla yapılmalı: gerçek kullanıcı sorularından 30–100 tanesi iyi bir başlangıçtır.

In [ ]:
eval_set = [
    ("Beş yıldan fazla kıdemim var, kaç gün izin hakkım olur?",       "izin_politikasi.md"),
    ("Kullanmadığım izinler seneye geçer mi?",                         "izin_politikasi.md"),
    ("Babalık izni kaç gün?",                                           "izin_politikasi.md"),
    ("Evden çalışırken hangi saatlerde ulaşılabilir olmalıyım?",       "uzaktan_calisma.md"),
    ("İnternet faturam için şirket destek veriyor mu?",                "uzaktan_calisma.md"),
    ("Akşam yemeği için şehir dışında ne kadar harcayabilirim?",       "masraf_politikasi.md"),
    ("Taksi parasını ne zaman geri alabilirim?",                        "masraf_politikasi.md"),
    ("Oltalama e-postası gelirse ne yapmalıyım?",                       "bilgi_guvenligi_2025.md"),
    ("USB bellek kullanabilir miyim?",                                  "bilgi_guvenligi_2025.md"),
    ("Sertifika sınavında kalırsam ücreti kim öder?",                   "egitim_gelisim.md"),
    ("Pahalı bir eğitimden sonra istifa edersem ne olur?",              "egitim_gelisim.md"),
    ("Yeni başlayanlar hangi eğitimleri almak zorunda?",                "egitim_gelisim.md"),
]

def evaluate(rag: AdvancedRAG, mode: str, k: int = 3, filters=None) -> Dict[str, float]:
    hits, rr = [], []
    for q, gold in eval_set:
        ids = rag.retrieve(q, mode=mode, k=k, filters=filters)["ids"]
        sources = [rag.chunks[i].metadata["source"] for i in ids]
        rank = next((r for r, s in enumerate(sources, 1) if s == gold), None)
        hits.append(rank is not None)
        rr.append(1 / rank if rank else 0.0)
    return {f"Hit@{k}": np.mean(hits), "MRR": np.mean(rr)}

results = {}
for mode in ["bm25", "dense", "hybrid", "full"]:
    results[mode] = evaluate(rag, mode, k=3)
pd.DataFrame(results).T.rename(index={"full": "hybrid + rerank + MMR (+LLM sorgu)"}).round(3)

,Hit@3,MRR
bm25,1.0,1.0
dense,1.0,1.0
hybrid,1.0,1.0
hybrid + rerank + MMR (+LLM sorgu),1.0,1.0


### Cevap sadakati (LLM-as-judge)
Cevaptaki her iddianın bağlamda gerçekten geçip geçmediğini ikinci bir LLM çağrısıyla puanlarız. Halüsinasyonu ölçmenin pratik bir yoludur. Yerel LLM gerekir; hakem olarak 7B veya üzeri bir model daha güvenilir puan verir.

In [ ]:
def judge_faithfulness(rag: AdvancedRAG, question: str) -> Optional[dict]:
    if not rag.llm.available:
        print("LLM yok, sadakat değerlendirmesi atlandı.")
        return None
    res = rag.answer(question, verbose=False)
    ctx = "\n\n".join(h["context"] for h in res["sources"])
    prompt = f"""Bağlam:
{ctx}

Cevap:
{res['answer']}

Cevaptaki her iddia bağlam tarafından destekleniyor mu? Sadece şu JSON'u döndür:
{{"puan": <0-1 arası sayı>, "desteklenmeyen_iddialar": [<metinler>]}}"""
    raw = rag.llm.complete(prompt, max_tokens=300)
    try:
        verdict = json.loads(re.search(r"\{.*\}", raw, re.S).group(0))
    except Exception:
        verdict = {"ham_cevap": raw}
    print(f"❓ {question}\n💬 {res['answer']}\n⚖️ {verdict}\n")
    return verdict

for q, _ in eval_set[:3]:
    judge_faithfulness(rag, q)

❓ Beş yıldan fazla kıdemim var, kaç gün izin hakkım olur?
💬 Beş yıldan fazla kıdemim varsa, yılda 26 iş günü izin hakkım olur. [2]
⚖️ {'puan': 1, 'desteklenmeyen_iddialar': []}

❓ Kullanmadığım izinler seneye geçer mi?
💬 Kullanılmayan izinlerin en fazla 5 gün bir sonraki yıla devredilebilir. Ancak, bu devreden izinler 31 Mart tarihine kadar kullanılmazsa yanar. Bu bilgi [1] kaynakından elde edilmiştir.
⚖️ {'puan': 0.8, 'desteklenmeyen_iddialar': ['Ancak, bu devreden izinler 31 Mart tarihine kadar kullanılmazsa yanar.']}

❓ Babalık izni kaç gün?
💬 Babalık izni 10 iş günüdür. [1], [2]
⚖️ {'puan': 1, 'desteklenmeyen_iddialar': []}



## 13. Kendi verinle kullanmak
```python
cfg = RAGConfig(chunk_size=200, chunk_overlap=40)      # gerçek belgeler için daha büyük parçalar
rag = AdvancedRAG(cfg)
rag.add_documents(load_directory("belgelerim/", extra_metadata={"category": "genel"}))
rag.save("benim_indeksim")

# Sonraki seferlerde:
rag = AdvancedRAG.load("benim_indeksim")
rag.answer("...", filters={"source": "sozlesme.pdf"})
```

### Ayar önerileri
| Durum | Ne değiştir? |
|---|---|
| Cevap eksik geliyor | `k_final` ↑, `neighbor_window` ↑, `chunk_size` ↑ |
| Alakasız parçalar geliyor | `use_rerank=True`, `k_final` ↓, `mmr_lambda` ↑ |
| Kod, ürün adı, rakam aramaları kaçıyor | `sparse_weight` ↑ |
| Kısa/belirsiz sorular kötü | `use_multi_query=True`, `use_hyde=True` |
| Daha yüksek kalite (daha yavaş) | `embed_model="BAAI/bge-m3"`, `rerank_model="BAAI/bge-reranker-v2-m3"` |
| Çok büyük veri (milyonlarca parça) | FAISS `IndexHNSWFlat` / Qdrant, Chroma gibi bir vektör veritabanı |

### 🚀 Kendi PDF Dosyalarınızı Yükleyin ve Sorgulayın

Colab ortamına yüklediğiniz PDF dosyalarını aşağıdaki kodu çalıştırarak RAG sistemine indeksleyebilirsiniz:

In [ ]:
# Dizindeki tüm yüklenen PDF dosyalarını otomatik bul ve RAG sistemine yükle
import glob

pdf_files = glob.glob("/content/*.pdf")

if pdf_files:
    print(f"{len(pdf_files)} adet PDF dosyası bulundu. Yükleniyor...")
    for pdf_path in pdf_files:
        try:
            pdf_docs = load_file(pdf_path)
            # Her dökümana hangi dosyadan geldiğini belirten etiket ekleyelim
            for d in pdf_docs:
                d.metadata["category"] = "kullanici_yuklemesi"
            rag.add_documents(pdf_docs)
            print(f"  ✅ {os.path.basename(pdf_path)} başarıyla indekslendi.")
        except Exception as e:
            print(f"  ❌ {os.path.basename(pdf_path)} yüklenirken hata oluştu: {e}")
    print("\nTüm belgeler başarıyla RAG sistemine entegre edildi!")
else:
    print("Klasörde yüklü PDF dosyası bulunamadı. Lütfen sol menüden PDF dosyalarınızı sürükleyip bırakarak yükleyin.")

7 adet PDF dosyası bulundu. Yükleniyor...
  ✅ 6566f16690e0a4da74d9fa55f729f124.pdf başarıyla indekslendi.
  ✅ cab135f209f5c13f01fd48b794eb5f34.pdf başarıyla indekslendi.


INFO | 134 belge → 949 parça eklendi (toplam 967) | embedding: sentence-transformers


  ✅ siemens-integrity-initiative-annual-report-2025-final.pdf başarıyla indekslendi.
  ✅ ff10e1e411735ba7e5043ef25dc1e8e0.pdf başarıyla indekslendi.
  ✅ 1c3b22ba5250b9df5fa353ae0a1d8934.pdf başarıyla indekslendi.
  ✅ c693dce324756cada8fd56af7755cde9.pdf başarıyla indekslendi.
  ✅ f38b54ab5da9fb7c4bf4d54dacf16638.pdf başarıyla indekslendi.

Tüm belgeler başarıyla RAG sistemine entegre edildi!


In [ ]:
# Yüklediğiniz PDF'ler hakkında test sorusu sorun:
if len(glob.glob("/content/*.pdf")) > 0:
    # Bu soruyu yüklediğiniz belgenin içeriğine göre değiştirebilirsiniz:
    _ = rag.answer("Yüklenen belgelerin ana konuları ve sunduğu önemli bilgiler nelerdir?")
else:
    print("Lütfen önce yukarıdaki hücreyi çalıştırarak PDF belgelerinizi indeksleyin.")

❓ Yüklenen belgelerin ana konuları ve sunduğu önemli bilgiler nelerdir?
↪ Sorgu varyantları: Numara: 1 | Soru: Yüklü olan belgelerde ele alınan ana konular ve temel bilgiler ne olabilir? | Numara: 2

💬 Yüklenen belgelerin ana konuları ve sunduğu önemli bilgiler şunlardır:

1. **Seyahat ve Masraf Politikası**:
   - Tüm iş amaçlı masraflar, harcama tarihinden itibaren en geç 30 gün içinde masraf sistemine girilmelidir.
   - Her masraf kalemi için fatura veya fiş yüklenmesi zorunludur; belgesiz masraflar ödenmez.
   - Şehir içi yemek masraflarında kişi başı günlük limit 600 TL, şehir dışı seyahatlerde 900 TL'dir.
   - Konaklamada dört yıldız üzeri oteller için önceden finans onayı alınmalıdır.
   - Onay akışı: 5.000 TL'ye kadar olan masraflar doğrudan yönetici onayları gerektirir; 5.000 TL üzerindeki masraflar için ek olarak finans direktörünün onayı gerekir.
   - Taxi masrafları yalnızca 22:00 sonrası veya toplu taşımanın mümkün olmadığı durumlarda karşılanır.

2. **Bilgi Güvenliği Polit

### 💬 RAG Test İstasyonu (Soru-Cevap Sınaması)

Aşağıdaki alana yüklediğiniz PDF belgelerinin içerikleriyle ilgili spesifik sorular yazarak RAG sisteminin doğruluğunu, kaynak gösterme yeteneğini ve doğruluğunu test edebilirsiniz:

In [ ]:
# Sorunuzu buraya yazın:
test_sorusu = "Yüklenen belgelerde geçen en kritik kural veya yönerge nedir?"

# Eğer RAGConfig veya diğer yapılar tanımlı değilse, önceki tanımları içeren hücreleri taklit edelim
if 'RAGConfig' not in globals():
    print("⚙️ Gerekli sınıflar ve tanımlar bulunamadı. Üst hücrelerin tanımları yükleniyor...")
    # İhtiyacımız olan tüm tanımlamaları yapıyoruz
    import os, re, math, json, glob, pickle, logging, warnings
    from dataclasses import dataclass, field, asdict
    from typing import Any, Callable, Dict, List, Optional
    from collections import Counter, defaultdict
    import numpy as np
    import pandas as pd

# Eğer RAG başlatılmadıysa otomatik başlat ve PDF'leri indeksle
if 'rag' not in globals():
    print("⚙️ RAG sistemi kuruluyor ve başlatılıyor...")
    cfg = RAGConfig()
    rag = AdvancedRAG(cfg)

    # PDF'leri tara ve ekle
    pdf_files = glob.glob("/content/*.pdf")
    if pdf_files:
        print(f"📂 {len(pdf_files)} adet PDF dosyası bulunup indeksleniyor...")
        for pdf_path in pdf_files:
            try:
                pdf_docs = load_file(pdf_path)
                for d in pdf_docs:
                    d.metadata["category"] = "kullanici_yuklemesi"
                rag.add_documents(pdf_docs)
            except Exception as e:
                print(f"  ❌ {os.path.basename(pdf_path)} yüklenemedi: {e}")
    else:
        print("⚠️ Yüklü PDF bulunamadı, varsayılan kurgusal belgeler yükleniyor...")
        rag.add_documents(documents)

# Cevabı ve kaynakları getir
print("\n🔍 RAG sistemi çalıştırılıyor...\n")
_ = rag.answer(test_sorusu)


🔍 RAG sistemi çalıştırılıyor...

❓ Yüklenen belgelerde geçen en kritik kural veya yönerge nedir?
↪ Sorgu varyantları: #1: Yüklenen tüm belgelerde tekrarlanan en önemli kurallar veya yönergeler hangileridir? | #2: Yüklenen her belgede en çok geçiren veya en kritik olan kurallar veya yönergeleri belirtin. | #3: Tüm yüklenen belgelerde en sık ve en kritik olarak geçtiği bilinen kurallar veya yönergeleri listele.

💬 Yüklenen belgelerde geçen en kritik kural veya yönerge, bilgi güvenliği politikasındaki "Gizli ve Çok Gizli veriler şifrelenmeden e-posta ile gönderilemez" ifadesidir. Bu kural, verilerin güvenliğini ve korumasını sağlamak için hassas bilgilerin şifrelenmemiş halde e-posta ile gönderilmesini yasaklamaktadır.

📚 Kaynaklar:
  [1] masraf_politikasi.md       skor=+0.005  Her masraf kalemi için fatura veya fiş yüklenmesi zorunludur; belgesiz...
  [2] masraf_politikasi.md       skor=+0.004  Seyahat ve Masraf Politikası İş amaçlı tüm masraflar, harcama tarihind...
  [3] bilgi_guvenli

### 🎯 Kendi Sorunuzu Sorun
Aşağıdaki hücreye kendi belgenizle ilgili merak ettiğiniz soruyu yazıp çalıştırabilirsiniz:

In [ ]:
# Sorunuzu tırnak işaretleri arasına yazın:
soru = "1 yıllık çalışansam yıllık iznim ne kadar"

# Cevabı getir:
_ = rag.answer(soru)

❓ 1 yıllık çalışansam yıllık iznim ne kadar
↪ Sorgu varyantları: **Açıklama:** Çalışma süresine bağlı olarak yıllık iznimin miktarını sormaktasınız. | yıllık çalıştığınızda yıllık iznim kaç gün olur? | **Açıklama:** Sorunuz, 1 yıl boyunca çalıştığınız durumunda alacağınız yıllık iznimizin süre uzunluğudur.

💬 1 yıllık çalışansanız yıllık ücretli izninizi 14 iş günü alabilirsiniz. [1], [2]

📚 Kaynaklar:
  [1] izin_politikasi.md         skor=+0.946  Yıllık İzin Politikası Çalışanlar kıdemlerine göre yıllık ücretli izin...
  [2] izin_politikasi.md         skor=+0.320  Beş yıldan fazla, on beş yıldan az kıdemi olanlara 20 iş günü, on beş ...
  [3] siemens-integrity-initiative-annual-report-2025-final.pdf#s99 skor=+0.000  Training length 33% 1% 66% 99 Short-Term Training Longer-Term Training...
  [4] masraf_politikasi.md       skor=+0.000  Seyahat ve Masraf Politikası İş amaçlı tüm masraflar, harcama tarihind...
-------------------------------------------------------------------------------